# V1-06: 기업 식별 인덱스 성능 비교

`05`에서 확인한 정규화·별칭·상장 종목 우선 규칙의 결과를 유지하면서, 매 질의 전수 탐색 비용을 줄일 수 있는지 비교합니다.

- A. 매 질의마다 118,000여 개 기업코드를 정규화하며 전수 탐색
- B. 시작 시 정규화 이름 인덱스를 한 번 만들고, 이후에는 인덱스에서 조회

LLM·Agent·새 패키지는 사용하지 않습니다. 이 notebook은 기업 식별의 의미 품질이 아니라, 같은 규칙을 더 효율적으로 실행할 수 있는지 확인하는 성능 비교입니다.

## 비교 기준

- 두 방식의 선택 종목코드·보류 결과가 같은지
- Entity Accuracy, Candidate Recall, Coverage, Unsafe Selection Rate가 같은지
- warm-up 후 p50/p95 질의 지연시간
- tracing 없이 측정한 인덱스 생성 시간과, 별도 측정한 인덱스 생성 중 Python 할당 peak

`tracemalloc` peak는 인덱스 생성 중 Python 할당량만 측정합니다. 전체 프로세스 RAM이나 모델 VRAM 사용량은 아닙니다.

In [1]:
import io
import os
import re
import sys
import time
import tracemalloc
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path
from statistics import median

In [2]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

api_key = os.environ.get('OPENDART_API_KEY')
if not api_key:
    raise ValueError('.env에 OPENDART_API_KEY를 설정해 주세요.')

In [3]:
sys.path.insert(0, str(project_root / 'src'))

from dart.client import CORP_CODE_URL, _get_bytes

In [4]:
def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    zip_stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(zip_stream):
        error_text = zip_bytes.decode('utf-8', errors='replace')
        raise RuntimeError(f'기업코드 응답이 ZIP이 아닙니다: {error_text}')
    with zipfile.ZipFile(zip_stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])

    root = ET.fromstring(xml_bytes)
    return [
        {child.tag: child.text or '' for child in item}
        for item in root.findall('list')
    ]

corp_records = load_corp_records(api_key)
print(f'기업코드 레코드 수: {len(corp_records):,}')

기업코드 레코드 수: 118,831


## 1. 동일한 개발 사례와 공통 선택 규칙

이전 notebook과 같은 개발 사례를 사용합니다. 따라서 이번 결과는 별칭 일반화 성능이 아니라, 동일한 규칙의 실행 방식 비교입니다.

In [5]:
TEST_CASES = [
    {'유형': '정확 법인명', '질문 기업명': '삼성전자', '정답 종목코드': '005930'},
    {'유형': '정확 법인명', '질문 기업명': 'SK하이닉스', '정답 종목코드': '000660'},
    {'유형': '공백 변형', '질문 기업명': 'SK 하이닉스', '정답 종목코드': '000660'},
    {'유형': '공백 변형', '질문 기업명': '현대 자동차', '정답 종목코드': '005380'},
    {'유형': '대소문자 변형', '질문 기업명': 'naver', '정답 종목코드': '035420'},
    {'유형': '별칭', '질문 기업명': '네이버', '정답 종목코드': '035420'},
    {'유형': '동일 법인명', '질문 기업명': '셀트리온', '정답 종목코드': '068270'},
    {'유형': '정확 법인명', '질문 기업명': '리노공업', '정답 종목코드': '058470'},
    {'유형': '존재하지 않는 기업', '질문 기업명': 'FinSight가상기업', '정답 종목코드': None},
]

def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())

ALIASES = {normalize_name('네이버'): normalize_name('NAVER')}

def canonical_query(query):
    normalized_query = normalize_name(query)
    return ALIASES.get(normalized_query, normalized_query)

def rank_candidates(candidates):
    return sorted(
        candidates,
        key=lambda record: (bool(record['stock_code'].strip()), record['modify_date']),
        reverse=True,
    )

def select_listed_if_unique(candidates):
    listed_candidates = [record for record in candidates if record['stock_code'].strip()]
    if len(listed_candidates) == 1:
        return listed_candidates[0]
    return candidates[0] if len(candidates) == 1 else None

## 2. 인덱스 생성 비용

동일 레코드가 한글·영문 이름 키가 같아도 한 번만 저장되도록 `corp_code` 기준으로 중복을 제거합니다.

In [6]:
def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(records_by_code.values()) for key, records_by_code in index.items()}

build_started_at = time.perf_counter()
name_index = build_name_index(corp_records)
index_build_ms = (time.perf_counter() - build_started_at) * 1_000

tracemalloc.start()
memory_probe_index = build_name_index(corp_records)
_, index_peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
del memory_probe_index

print(f'정규화 이름 키 수: {len(name_index):,}')
print(f'인덱스 생성 시간(tracing 제외): {index_build_ms:.1f} ms')
print(f'별도 인덱스 생성 중 Python 할당 peak: {index_peak_bytes / 1024 / 1024:.2f} MiB')

정규화 이름 키 수: 218,777
인덱스 생성 시간(tracing 제외): 772.7 ms
별도 인덱스 생성 중 Python 할당 peak: 84.83 MiB


## 3. 전수 탐색과 인덱스 탐색 비교

각 질의는 warm-up 3회 후 30회 반복합니다. ZIP 다운로드·XML 파싱·인덱스 생성 시간은 질의 지연시간에 포함하지 않습니다.

In [7]:
def scan_candidates(query):
    key = canonical_query(query)
    return [
        record for record in corp_records
        if key in {normalize_name(record['corp_name']), normalize_name(record['corp_eng_name'])}
    ]

def indexed_candidates(query):
    return name_index.get(canonical_query(query), [])

WARMUP_ITERATIONS = 3
MEASURE_ITERATIONS = 30

def evaluate_strategy(strategy_name, candidate_function):
    results = []
    latency_samples = []
    for case in TEST_CASES:
        for _ in range(WARMUP_ITERATIONS):
            select_listed_if_unique(rank_candidates(candidate_function(case['질문 기업명'])))

        case_latencies = []
        for _ in range(MEASURE_ITERATIONS):
            started_at = time.perf_counter()
            candidates = rank_candidates(candidate_function(case['질문 기업명']))
            selected = select_listed_if_unique(candidates)
            case_latencies.append((time.perf_counter() - started_at) * 1_000)

        selected_stock_code = selected['stock_code'] if selected else None
        selected_corp_code = selected['corp_code'] if selected else None
        candidate_stock_codes = [record['stock_code'] for record in candidates if record['stock_code'].strip()]
        results.append({
            **case,
            '전략': strategy_name,
            '후보 수': len(candidates),
            '후보 corp_code': [record['corp_code'] for record in candidates],
            '선택 corp_code': selected_corp_code,
            '선택 종목코드': selected_stock_code,
            '정답 여부': selected_stock_code == case['정답 종목코드'],
            '후보 포함 여부': case['정답 종목코드'] in candidate_stock_codes if case['정답 종목코드'] else not candidates,
            '반복 중앙 지연시간(ms)': median(case_latencies),
        })
        latency_samples.extend(case_latencies)
    return results, latency_samples

scan_results, scan_latencies = evaluate_strategy('정규화 전수 탐색', scan_candidates)
index_results, index_latencies = evaluate_strategy('정규화 인덱스 탐색', indexed_candidates)

for scan_row, index_row in zip(scan_results, index_results):
    assert scan_row['후보 corp_code'] == index_row['후보 corp_code'], (scan_row, index_row)
    assert scan_row['선택 corp_code'] == index_row['선택 corp_code'], (scan_row, index_row)

print('모든 사례에서 두 방식의 후보 corp_code 목록과 선택 corp_code가 같습니다.')

모든 사례에서 두 방식의 후보 corp_code 목록과 선택 corp_code가 같습니다.


In [8]:
def percentile(values, percentile_value):
    ordered = sorted(values)
    return ordered[round((len(ordered) - 1) * percentile_value)]

def summarize_results(results, latencies):
    positive_rows = [row for row in results if row['정답 종목코드'] is not None]
    selected = [row for row in positive_rows if row['선택 종목코드'] is not None]
    selected_all = [row for row in results if row['선택 종목코드'] is not None]
    wrong_selected = [row for row in selected_all if not row['정답 여부']]
    safety_rows = [row for row in results if row['정답 종목코드'] is None]
    return {
        '전략': results[0]['전략'],
        'Entity Accuracy': sum(row['정답 여부'] for row in positive_rows) / len(positive_rows),
        'Candidate Recall': sum(row['후보 포함 여부'] for row in positive_rows) / len(positive_rows),
        'Coverage': len(selected) / len(positive_rows),
        'Unsafe Selection Rate': len(wrong_selected) / len(selected_all) if selected_all else 0.0,
        '보류 정확도': sum(row['정답 여부'] for row in safety_rows) / len(safety_rows),
        'p50 지연시간(ms)': median(latencies),
        'p95 지연시간(ms)': percentile(latencies, 0.95),
    }

for summary in [summarize_results(scan_results, scan_latencies), summarize_results(index_results, index_latencies)]:
    print({key: round(value, 4) if isinstance(value, float) else value for key, value in summary.items()})

{'전략': '정규화 전수 탐색', 'Entity Accuracy': 1.0, 'Candidate Recall': 1.0, 'Coverage': 1.0, 'Unsafe Selection Rate': 0.0, '보류 정확도': 1.0, 'p50 지연시간(ms)': 207.4628, 'p95 지연시간(ms)': 212.0438}
{'전략': '정규화 인덱스 탐색', 'Entity Accuracy': 1.0, 'Candidate Recall': 1.0, 'Coverage': 1.0, 'Unsafe Selection Rate': 0.0, '보류 정확도': 1.0, 'p50 지연시간(ms)': 0.0013, 'p95 지연시간(ms)': 0.0016}


## 4. 실행 후 해석

- 선택 결과와 품질 지표가 같고 인덱스의 p50/p95가 명확히 낮으면, 다음 기업 식별 후보에서는 인덱스 탐색을 유지합니다.
- 인덱스 생성 시간·Python 할당 peak는 애플리케이션 시작 비용입니다. 이후 질의 수가 많을수록 인덱스 비용을 회수하기 쉽습니다.
- 이 비교는 동일한 정규화 규칙의 실행 방식만 다룹니다. 별칭·오타·약칭 품질은 다음 유사도 후보 비교와 더 큰 holdout 평가셋에서 판단합니다.